In [4]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

import json
import warnings
warnings.filterwarnings('ignore')

In [5]:
train_df = pd.read_parquet('../data/raw/train_df.parquet', engine='fastparquet')
unseen_df = pd.read_parquet('../data/raw/unseen_df.parquet', engine='fastparquet')

with open('../data/processed/final_features.json', 'r') as f:
    final_features = json.load(f)


In [6]:
x = train_df[final_features]
y = train_df['target']

We noticed `-99999` in some columns so its time to handle them as null values.

`-99999` likely represents missing or unavailable information.

In [7]:
x = x.replace(-99999, np.nan)
unseen_df = unseen_df.replace(-99999, np.nan)

In [8]:
missing_values = x.isnull().sum().sort_values(ascending=False)
missing_values.head(10)

CC_enq_L12m                  6321
PL_enq_L12m                  6321
enq_L3m                      6321
time_since_recent_enq        6321
time_since_recent_payment    4291
Age_Newest_TL                  40
Age_Oldest_TL                  40
CC_Flag                         0
CC_TL                           0
EDUCATION                       0
dtype: int64

### Handling Missing Values

In [9]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 51336 entries, 0 to 51335
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age_Newest_TL              51296 non-null  float64
 1   Age_Oldest_TL              51296 non-null  float64
 2   CC_Flag                    51336 non-null  int64  
 3   CC_TL                      51336 non-null  int64  
 4   CC_enq_L12m                45015 non-null  float64
 5   EDUCATION                  51336 non-null  object 
 6   GENDER                     51336 non-null  object 
 7   GL_Flag                    51336 non-null  int64  
 8   HL_Flag                    51336 non-null  int64  
 9   Home_TL                    51336 non-null  int64  
 10  MARITALSTATUS              51336 non-null  object 
 11  NETMONTHLYINCOME           51336 non-null  int64  
 12  Other_TL                   51336 non-null  int64  
 13  PL_Flag                    51336 non-null  int64  
 14  P

In [10]:
num_cols = x.select_dtypes(include=['int', 'float']).columns
num_cols

Index(['Age_Newest_TL', 'Age_Oldest_TL', 'CC_Flag', 'CC_TL', 'CC_enq_L12m',
       'GL_Flag', 'HL_Flag', 'Home_TL', 'NETMONTHLYINCOME', 'Other_TL',
       'PL_Flag', 'PL_TL', 'PL_enq_L12m', 'Secured_TL', 'Time_With_Curr_Empr',
       'Tot_Missed_Pmnt', 'Tot_TL_closed_L12M', 'Unsecured_TL', 'enq_L3m',
       'max_recent_level_of_deliq', 'num_dbt', 'num_dbt_12mts',
       'num_deliq_6_12mts', 'num_lss', 'num_std_12mts', 'num_sub',
       'num_sub_12mts', 'num_sub_6mts', 'num_times_60p_dpd',
       'pct_CC_enq_L6m_of_ever', 'pct_PL_enq_L6m_of_ever',
       'pct_tl_closed_L12M', 'pct_tl_closed_L6M', 'pct_tl_open_L6M',
       'recent_level_of_deliq', 'time_since_recent_enq',
       'time_since_recent_payment'],
      dtype='str')

In [11]:
median_imputer = SimpleImputer(strategy='median') # we chose median because its robust to outliers
x[num_cols] = median_imputer.fit_transform(x[num_cols])
unseen_df[num_cols] = median_imputer.transform(unseen_df[num_cols])

In [12]:
x.isnull().sum()

Age_Newest_TL                0
Age_Oldest_TL                0
CC_Flag                      0
CC_TL                        0
CC_enq_L12m                  0
EDUCATION                    0
GENDER                       0
GL_Flag                      0
HL_Flag                      0
Home_TL                      0
MARITALSTATUS                0
NETMONTHLYINCOME             0
Other_TL                     0
PL_Flag                      0
PL_TL                        0
PL_enq_L12m                  0
Secured_TL                   0
Time_With_Curr_Empr          0
Tot_Missed_Pmnt              0
Tot_TL_closed_L12M           0
Unsecured_TL                 0
enq_L3m                      0
first_prod_enq2              0
last_prod_enq2               0
max_recent_level_of_deliq    0
num_dbt                      0
num_dbt_12mts                0
num_deliq_6_12mts            0
num_lss                      0
num_std_12mts                0
num_sub                      0
num_sub_12mts                0
num_sub_

### Encoding

In [13]:
cat_cols = x.select_dtypes(include=['object']).columns
cat_cols

Index(['EDUCATION', 'GENDER', 'MARITALSTATUS', 'first_prod_enq2',
       'last_prod_enq2'],
      dtype='str')

In [14]:
for col in cat_cols:
    print(x[col].value_counts())
    print()

EDUCATION
GRADUATE          16673
12TH              14467
SSC                9276
UNDER GRADUATE     5492
OTHERS             2917
POST-GRADUATE      2242
PROFESSIONAL        269
Name: count, dtype: int64

GENDER
M    45245
F     6091
Name: count, dtype: int64

MARITALSTATUS
Married    37752
Single     13584
Name: count, dtype: int64

first_prod_enq2
others          28120
ConsumerLoan    11860
PL               4889
AL               2870
CC               2188
HL               1409
Name: count, dtype: int64

last_prod_enq2
others          20831
ConsumerLoan    17793
PL               7959
CC               2339
AL               1511
HL                903
Name: count, dtype: int64



We are using OHE because these are nominal columns, there is no natural order.

In [15]:
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(x[cat_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(cat_cols),
    index=x.index
)

x = x.drop(columns=cat_cols)
x = pd.concat([x, encoded_df], axis=1)

In [16]:
encoded_unseen = encoder.transform(unseen_df[cat_cols])

encoded_unseen_df = pd.DataFrame(
    encoded_unseen,
    columns=encoder.get_feature_names_out(cat_cols),
    index=unseen_df.index
)

unseen_df = unseen_df.drop(columns=cat_cols)
unseen_df = pd.concat([unseen_df, encoded_unseen_df], axis=1)

In [17]:
print(x.shape)

(51336, 55)


In [18]:
y.value_counts(normalize=True)

target
1    0.74026
0    0.25974
Name: proportion, dtype: float64

Since class is unbalanced, we may need to use `scale_pos_weight` or `class_weight`, in modelling phase.

In [19]:
processed_df = pd.concat([x, y], axis=1)

processed_df.to_parquet("../data/processed/model_ready_data.parquet", index=False)

In [20]:
unseen_df.to_parquet("../data/processed/unseen_processed.parquet", index=False)

In [21]:
unseen_df

,pct_tl_open_L6M,pct_tl_closed_L6M,Tot_TL_closed_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,CC_TL,Home_TL,PL_TL,Secured_TL,Unsecured_TL,...,first_prod_enq2_CC,first_prod_enq2_ConsumerLoan,first_prod_enq2_HL,first_prod_enq2_PL,first_prod_enq2_others,last_prod_enq2_CC,last_prod_enq2_ConsumerLoan,last_prod_enq2_HL,last_prod_enq2_PL,last_prod_enq2_others
0,0.000,0.0,0.0,0.000,0.0,0.0,0.0,4.0,1.0,4.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.125,0.0,0.0,0.000,1.0,0.0,0.0,0.0,2.0,6.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000,0.0,1.0,0.167,0.0,0.0,0.0,0.0,6.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.000,0.0,0.0,0.000,0.0,0.0,0.0,1.0,1.0,5.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
96,0.222,0.0,2.0,0.222,0.0,0.0,0.0,0.0,1.0,8.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
97,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
98,0.000,1.0,1.0,1.000,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
